# 🧠 Online Retail Customer Segmentation
## OASIS INFOBYTE SIP — Data Analytics Level 1, Task 2

This notebook analyses online retail purchasing behaviour and segments customers using **RFM analysis** and **K-Means clustering**.

Every major analytical stage produces a visualisation, and charts are saved in the project's `outputs/` folder.

## 🎯 Objectives
- Clean and validate transaction data.
- Explore sales trends, products and countries.
- Calculate customer-level Recency, Frequency and Monetary (RFM) measures.
- Visualise RFM distributions.
- Select clusters using the Elbow Method and silhouette score.
- Apply K-Means and visualise customer segments.
- Translate findings into business recommendations.

## 1. Dataset source
The dataset is the **Online Retail** dataset from the UCI Machine Learning Repository (Dataset 352), covering transactions from 1 December 2010 to 9 December 2011.

Citation: Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository. DOI: 10.24432/C5BW33.

In [ ]:
# Install once if needed:
# %pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
OUTPUTS = Path('../outputs')
OUTPUTS.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)

## 2. Load the UCI dataset

In [ ]:
online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()
print('Shape:', df.shape)
display(df.head())

## 3. Data quality checks and cleaning

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Negative quantities:', (df['Quantity'] < 0).sum())
print('Non-positive prices:', (df['UnitPrice'] <= 0).sum())
print('Cancelled invoices:', df['InvoiceNo'].astype(str).str.upper().str.startswith('C').sum())

In [ ]:
df = df.drop_duplicates().copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
clean = df[~df['InvoiceNo'].astype(str).str.upper().str.startswith('C') & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['CustomerID'].notna() & df['InvoiceDate'].notna()].copy()
clean['CustomerID'] = clean['CustomerID'].astype(int).astype(str)
clean['Revenue'] = clean['Quantity'] * clean['UnitPrice']
print('Cleaned shape:', clean.shape)
display(clean.head())

## 4. 📊 Exploratory analysis

In [ ]:
monthly_sales = clean.set_index('InvoiceDate')['Revenue'].resample('MS').sum()
fig, ax = plt.subplots(figsize=(12,5))
monthly_sales.plot(ax=ax, marker='o')
ax.set_title('Monthly Revenue Trend'); ax.set_xlabel('Month'); ax.set_ylabel('Revenue (£)')
plt.tight_layout(); fig.savefig(OUTPUTS/'monthly_revenue_trend.png', dpi=160, bbox_inches='tight'); plt.show()

### Monthly revenue trend
![Monthly revenue trend](../outputs/monthly_revenue_trend.png)

In [ ]:
top_products = clean.groupby('Description', dropna=False)['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10,6))
top_products.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Products by Revenue'); ax.set_xlabel('Revenue (£)')
plt.tight_layout(); fig.savefig(OUTPUTS/'top_10_products_by_revenue.png', dpi=160, bbox_inches='tight'); plt.show()

### Top products
![Top products](../outputs/top_10_products_by_revenue.png)

In [ ]:
country_sales = clean.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10,5))
country_sales.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Countries by Revenue'); ax.set_xlabel('Revenue (£)')
plt.tight_layout(); fig.savefig(OUTPUTS/'top_10_countries_by_revenue.png', dpi=160, bbox_inches='tight'); plt.show()

### Top markets
![Top countries](../outputs/top_10_countries_by_revenue.png)

## 5. 👥 Build the RFM table

In [ ]:
snapshot_date = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = clean.groupby('CustomerID').agg(Recency=('InvoiceDate', lambda x: (snapshot_date-x.max()).days), Frequency=('InvoiceNo','nunique'), Monetary=('Revenue','sum')).reset_index()
display(rfm.describe().round(2)); display(rfm.head())

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4))
for ax, column in zip(axes,['Recency','Frequency','Monetary']):
    sns.histplot(rfm[column], kde=True, ax=ax)
    ax.set_title(f'{column} Distribution')
plt.tight_layout(); fig.savefig(OUTPUTS/'rfm_distributions.png', dpi=160, bbox_inches='tight'); plt.show()

### RFM distributions
![RFM distributions](../outputs/rfm_distributions.png)

## 6. ⚙️ Prepare RFM features

In [ ]:
rfm_features = rfm[['Recency','Frequency','Monetary']].copy()
rfm_log = np.log1p(rfm_features)
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)
print('Customers available for segmentation:', len(rfm))

## 7. 📐 Select the number of clusters

In [ ]:
inertias=[]; silhouette_scores=[]; k_values=range(2,9)
for k in k_values:
    model=KMeans(n_clusters=k, random_state=42, n_init=20)
    labels=model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))
fig, axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(list(k_values),inertias,marker='o'); axes[0].set_title('Elbow Method'); axes[0].set_xlabel('Number of clusters (k)'); axes[0].set_ylabel('Inertia')
axes[1].plot(list(k_values),silhouette_scores,marker='o'); axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('Number of clusters (k)'); axes[1].set_ylabel('Silhouette score')
plt.tight_layout(); fig.savefig(OUTPUTS/'cluster_selection.png',dpi=160,bbox_inches='tight'); plt.show()
scores=pd.DataFrame({'k':list(k_values),'inertia':inertias,'silhouette':silhouette_scores}); display(scores.round(4))
best_k=int(scores.loc[scores['silhouette'].idxmax(),'k']); print('Highest silhouette score occurs at k =',best_k)

### Cluster-selection diagnostics
![Elbow and silhouette plots](../outputs/cluster_selection.png)

## 8. 🤖 Fit K-Means and profile the segments

In [ ]:
kmeans=KMeans(n_clusters=best_k,random_state=42,n_init=20)
rfm['Cluster']=kmeans.fit_predict(X)
cluster_profile=rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].agg(['mean','median','count']).round(2)
display(cluster_profile)

In [ ]:
cluster_counts=rfm['Cluster'].value_counts().sort_index()
fig,ax=plt.subplots(figsize=(8,5)); cluster_counts.plot(kind='bar',ax=ax)
ax.set_title('Customers per Segment'); ax.set_xlabel('Cluster'); ax.set_ylabel('Number of customers')
plt.tight_layout(); fig.savefig(OUTPUTS/'customers_per_segment.png',dpi=160,bbox_inches='tight'); plt.show()

### Segment sizes
![Customers per segment](../outputs/customers_per_segment.png)

In [ ]:
cluster_means=rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,column in zip(axes,['Recency','Frequency','Monetary']):
    cluster_means[column].plot(kind='bar',ax=ax)
    ax.set_title(f'Average {column} by Segment'); ax.set_xlabel('Cluster'); ax.set_ylabel(column)
plt.tight_layout(); fig.savefig(OUTPUTS/'rfm_by_segment.png',dpi=160,bbox_inches='tight'); plt.show()

### RFM comparison by segment
![RFM by segment](../outputs/rfm_by_segment.png)

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
for cluster in sorted(rfm['Cluster'].unique()):
    subset=rfm[rfm['Cluster']==cluster]
    ax.scatter(subset['Frequency'],subset['Monetary'],label=f'Cluster {cluster}',alpha=0.6)
ax.set_title('Customer Segments: Frequency vs Monetary Value'); ax.set_xlabel('Frequency (unique invoices)'); ax.set_ylabel('Monetary Value (£)'); ax.legend(title='Segment')
plt.tight_layout(); fig.savefig(OUTPUTS/'customer_segments_scatter.png',dpi=160,bbox_inches='tight'); plt.show()

### Customer segment map
![Customer segments](../outputs/customer_segments_scatter.png)

## 9. 💡 Business interpretation
Cluster numbers are arbitrary, so interpret segments from their RFM profiles.

- **High frequency + high monetary + low recency:** loyal/high-value customers — prioritise retention and personalised offers.
- **High monetary but lower frequency:** high-potential customers — encourage repeat purchases and cross-selling.
- **High recency + lower frequency/monetary:** at-risk or inactive customers — use reactivation campaigns.
- **High frequency but lower monetary:** frequent lower-value customers — use bundles and recommendations to increase basket value.

## 10. 🏁 Conclusion
The analysis combines exploratory data analysis, RFM feature engineering and unsupervised learning to identify meaningful groups of online retail customers. The resulting segments provide a foundation for targeted retention, reactivation and customer-value strategies.